In [ ]:
# Instalar bibliotecas necesarias
!pip install gspread oauth2client python-docx requests Pillow
!pip install gspread gspread-dataframe google-auth python-docx pillow pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.3/244.3 kB 4.7 MB/s eta 0:00:00


In [ ]:
# Importar librerías necesarias
import os
import requests
from docx import Document
from docx.shared import Pt
from docx.shared import Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from PIL import Image
from io import BytesIO
import pandas as pd
import gspread
from gspread_dataframe import get_as_dataframe
from google.colab import auth
from google.auth import default
import urllib.parse
from lxml.etree import QName as qn

# Autenticación automática con Google
auth.authenticate_user()
creds, _ = default()
client = gspread.authorize(creds)

# Conectar Google Drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:

# Abrir Google Sheets y leer datos
spreadsheet_id = '1bej0IDxNHjuveq80fG2LCmcoAKoH9GASOVmHyTeZJkE'  # Cambia este ID por el de tu hoja de cálculo
sheet = client.open_by_key(spreadsheet_id).worksheet('Total_') # Changed from sheet1 to worksheet('Total_')
ruta_drive = '/content/drive/MyDrive/QR_Completos'

# Refresh formulas in the sheet
sheet.batch_update([{
    'range': 'A1:O329',  # Adjust range to cover all your data if needed
    'values': [['']]  # Dummy update to trigger formula recalculation
}])

# Get the updated values after formula refresh
data = sheet.get_all_values()

# Create DataFrame from the updated values
datos_hoja = pd.DataFrame(data[1:], columns=data[0])

# Define a list of selections to process
selecciones = ['M1','M2','M3']

# Configuración de formato de texto
def agregar_texto_formateado(paragraph, texto, bold=True, font_size=11, alignment=WD_ALIGN_PARAGRAPH.JUSTIFY):
    run = paragraph.add_run(texto)
    run.bold = bold
    run.font.size = Pt(font_size)

     # Set font to Calibri
    run.font.name = 'Calibri'

    paragraph.alignment = alignment
    paragraph.paragraph_format.line_spacing = 1.08
    paragraph.paragraph_format.space_after = Pt(8)

# Modify the generar_qr_y_ficha function to accept the 'seleccion' parameter:
def generar_qr_y_ficha(row, doc, seleccion):  # Add 'seleccion' parameter
    # ... (your existing code)
    # Función para generar el QR y crear el documento en Word
    url = row['url']
    cursohorario = row['cursohorario']
    facultad = row['facultad']
    escuela = row['escuela']
    ciclo = row['ciclo']
    curso_nombre = row['cursonombre']
    horario = row['horario']
    nombre_docente = row['nombredocente']
    correo_docente = row['correodocente']
    alumnos = row['totalmatriculados']


    # Generar el QR usando la API pública de QuickChart.io
    encoded_url = urllib.parse.quote(url)
    qr_url = f'https://quickchart.io/qr?text={encoded_url}&size=450'
    response = requests.get(qr_url)

    if response.status_code == 200:
        image = Image.open(BytesIO(response.content))

        # Get image dimensions
        width, height = image.size

        # Crop the image - adjust values as needed
        crop_percentage = 0.07  # Example: 7% crop from each side
        left = int(width * crop_percentage)
        top = int(height * crop_percentage)
        right = width - int(width * crop_percentage)
        bottom = height - int(height * crop_percentage)

        cropped_image = image.crop((left, top, right, bottom))

        # Update the image path and document save path to use the 'seleccion'
        image_path = f'{ruta_drive}/QRs/{seleccion}/{cursohorario}.png'

        cropped_image.save(image_path)

        doc = Document()

        # Establecer el tamaño de página a A4
        section = doc.sections[0]  # Obtener la primera sección del documento

        # Configuración de márgenes para el documento individual
        sections = doc.sections
        for section in doc.sections:
          section.page_width = Inches(8.27)  # Ancho de A4 en pulgadas
          section.page_height = Inches(11.69)  # Alto de A4 en pulgadas
          section.top_margin = Pt(1.75 * 28.3465)  # 1.75 cm en puntos
          section.bottom_margin = Pt(2.5 * 28.3465)  # 2.5 cm en puntos
          section.left_margin = Pt(3 * 28.3465)  # 3 cm en puntos
          section.right_margin = Pt(3 * 28.3465)  # 3 cm en puntos


        # Crear una ficha para el QR actual
        doc.add_picture(image_path, width=Pt(425))
        agregar_texto_formateado(doc.add_paragraph(), f'FACULTAD: {facultad}')
        agregar_texto_formateado(doc.add_paragraph(), f'ESCUELA: {escuela}         CURSO: {curso_nombre}')
        agregar_texto_formateado(doc.add_paragraph(), f'HORARIO DEL CURSO: {horario}')
        agregar_texto_formateado(doc.add_paragraph(), f'DOCENTE: {nombre_docente}    CORREO: {correo_docente}')
        agregar_texto_formateado(doc.add_paragraph(), f'N° TOTAL DE ALUMNOS:     {alumnos}   N° DE ALUMNOS EN AULA: ________ ')
        agregar_texto_formateado(doc.add_paragraph(), f'RECHAZOS: _________                  N° DE MENORES DE EDAD:_______________')
        agregar_texto_formateado(doc.add_paragraph(), 'APLICADOR/A: ________________________________________________________________')
        agregar_texto_formateado(doc.add_paragraph(), 'FECHA: __________________________  HORA DE APLICACIÓN: ________________________')
        agregar_texto_formateado(doc.add_paragraph(), 'OBSERVACIONES:')
        agregar_texto_formateado(doc.add_paragraph(), '____________________________________________________________________________')
        agregar_texto_formateado(doc.add_paragraph(), '____________________________________________________________________________')
        agregar_texto_formateado(doc.add_paragraph(), {url}, bold=True, alignment=WD_ALIGN_PARAGRAPH.CENTER)

        # Guardar documento individual en Google Drive
        doc.save(f'{ruta_drive}/Fichas_word/{seleccion}/{cursohorario}.docx')


        print(f'Código QR y datos agregados para: {cursohorario}')
    else:
        print(f'Error al descargar el QR para {cursohorario}: {response.status_code}')


# Iterate through each selection
for seleccion in selecciones:
    # Filter data for the current selection
    datos_seleccion = datos_hoja[datos_hoja['muestra'] == seleccion]
    # Crear carpetas para guardar documentos en Google Drive
    os.makedirs(f'{ruta_drive}/QRs/{seleccion}', exist_ok=True)
    os.makedirs(f'{ruta_drive}/Fichas_word/{seleccion}', exist_ok=True)

# Process each row in the filtered data  (This is the part you asked about)
    for index, row in datos_seleccion.iterrows():
        doc = Document()
        generar_qr_y_ficha(row, doc, seleccion)  # Pass 'seleccion' to the function


Código QR y datos agregados para: Prueba
Código QR y datos agregados para: MEG03BTU
Código QR y datos agregados para: ME903AAE
Código QR y datos agregados para: AD201AAE
Código QR y datos agregados para: AD521AAE
Código QR y datos agregados para: AD206AAE
Código QR y datos agregados para: AD207AAE
Código QR y datos agregados para: AD801BAE
Código QR y datos agregados para: AD702BAE
Código QR y datos agregados para: FIG03AAO
Código QR y datos agregados para: MEG07AAO
Código QR y datos agregados para: IFG01AAO
Código QR y datos agregados para: AG101AAO
Código QR y datos agregados para: ME260AAO
Código QR y datos agregados para: AR502BAT
Código QR y datos agregados para: AR302AAT
Código QR y datos agregados para: IC307AAT
Código QR y datos agregados para: AR204AAT
Código QR y datos agregados para: AR405AAT
Código QR y datos agregados para: CBG04ABI
Código QR y datos agregados para: CB103BBI
Código QR y datos agregados para: QU130ABI
Código QR y datos agregados para: CB529ABI
Código QR y d

In [ ]:
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
import pandas as pd

folder_ids = ['1-BiNgC6DnDT7gCwnPKK4j-Hq6U95DseB', '1FlbgsA7ase9zBORDAZcJviymo1N_yCsL', '1T0TWJ5CYNzvqMC8vwzjJxRA8l6s4U9cn']

def get_shareable_link(file_id):
    """Gets the shareable link for a file."""
    try:
        drive_service = build('drive', 'v3', credentials=creds)
        file_metadata = drive_service.files().get(fileId=file_id, fields='webViewLink').execute()
        return file_metadata['webViewLink']
    except HttpError as error:
        print(f'An error occurred: {error}')
        return None

# Create an empty list to store the DataFrames
all_dfs = []

# Get shareable links for files and append DataFrames to the list
for folder_id in folder_ids:
    try:
        drive_service = build('drive', 'v3', credentials=creds)

        # Initialize variables for pagination
        page_token = None
        all_files = []

        # Loop through pages until no more pages are available
        while True:
            response = drive_service.files().list(
                q=f"'{folder_id}' in parents",
                fields='nextPageToken, files(id, name)',
                pageToken=page_token
            ).execute()

            # Add files from this page to the list
            all_files.extend(response.get('files', []))

            # Get next page token
            page_token = response.get('nextPageToken')

            # Break loop if no more pages
            if not page_token:
                break

        # Create a dictionary to store file names and shareable links
        shareable_links = {}
        for file in all_files:
            shareable_link = get_shareable_link(file.get('id'))
            if shareable_link:
                shareable_links[file.get('name')] = shareable_link

        # Create a DataFrame from the dictionary
        df_shareable_links = pd.DataFrame.from_dict(shareable_links, orient='index', columns=['Shareable Link'])
        df_shareable_links.index.name = 'curso-horario'
        df_shareable_links.index = df_shareable_links.index.str.replace('.docx', '', regex=False)

        # Append the DataFrame to the list
        all_dfs.append(df_shareable_links)

    except HttpError as error:
        print(f'An error occurred: {error}')

# Concatenate all DataFrames in the list into a single DataFrame
merged_df = pd.concat(all_dfs)

# Display the merged DataFrame
print(merged_df)

                                                  Shareable Link
curso-horario                                                   
MH504AMD       https://docs.google.com/document/d/1FdNjCBld3o...
MH305AMD       https://docs.google.com/document/d/1FA2Du4ppPA...
MH406AMD       https://docs.google.com/document/d/1FUneNo1vV2...
MH403AMD       https://docs.google.com/document/d/1FLteyrqtBF...
ML719AMT       https://docs.google.com/document/d/1FGQT_e76W0...
...                                                          ...
AD112BAE       https://docs.google.com/document/d/1ThjAt-Aj2q...
EC903ATU       https://docs.google.com/document/d/1TMHJG5a2NP...
ME903BAE       https://docs.google.com/document/d/1TPW-hr-44l...
AD111AAE       https://docs.google.com/document/d/1TY3h9J3eII...
IFG01AAE       https://docs.google.com/document/d/1TZeP6ScKwY...

[328 rows x 1 columns]


In [ ]:
# Assuming df_shareable_links is your DataFrame with shareable links
num_filas = len(merged_df)
print(f"El DataFrame df_shareable_links tiene {num_filas} filas.")

# Get the first shareable link from df_shareable_links
first_link = merged_df['Shareable Link'].iloc[0]  # Get the link from the first row

# Print the extracted link
print(first_link)

El DataFrame df_shareable_links tiene 328 filas.
https://docs.google.com/document/d/1FdNjCBld3oy-FGQAtarYcbIQtrOxttql/edit?usp=drivesdk&ouid=104569627089828962138&rtpof=true&sd=true


In [ ]:
# Assuming 'client' is your authenticated gspread client
# and 'new_spreadsheet_id' is the ID of your spreadsheet
from gspread_dataframe import set_with_dataframe
from gspread.exceptions import WorksheetNotFound

# 1. Check if worksheet exists, create if it doesn't
try:
    worksheet = client.open_by_key(spreadsheet_id).worksheet("Shareable Links")
except WorksheetNotFound:
    worksheet = client.open_by_key(spreadsheet_id).add_worksheet(title="Shareable Links", rows="394", cols="2")
    # Adjust rows and cols as needed

# 2. Reset index to include 'curso-horario' as a column
df_to_export = merged_df.reset_index()  # This line is added

# 3. Export the DataFrame to the worksheet
set_with_dataframe(worksheet, df_to_export)  # Export the modified DataFrame

print("DataFrame exported to 'Shareable Links' sheet.")

DataFrame exported to 'Shareable Links' sheet.
